In [2]:
from datasets import load_dataset

/home/miguel/miniconda3/envs/llm_agent/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
level_1_problems = load_dataset("gaia-benchmark/GAIA", "2023_level1", split="validation")
print(len(level_1_problems))

53


In [7]:
from pydantic import BaseModel

class GaiaOutput(BaseModel):
    is_solvable: bool # para indicar si el problema es resoluble o no
    unsolvable_reason: str = "" # para indicar la razón por la cual el problema no es resoluble
    final_answer: str = "" # para indicar la respuesta final del problema, si es resoluble

In [8]:
system_prompt = """
You are a general AI assistant. I will ask you a question.  First, determine if you can solve this problem with your current capabilities and set “is_solvable” accordingly. If you can solve it, set “is_solvable” to true and provide your answer in “final_answer”. If you cannot solve it, set “is_solvable” to false and explain why in “unsolvable_reason”.  Your final answer should be a number OR as few words as possible OR a commaseparated list of numbers and/or strings.  If you are asked for a number, don’t use a comma to write your number; also don’t use units such as $ or a percent sign unless specified otherwise.  If you are asked for a string, don’t use articles, neither abbreviations (e.g., for cities), and write the digits in plain text unless specified otherwise.  If you are asked for a comma-separated list, apply the above rules depending on whether the element is a number or a string.
"""

In [9]:
import asyncio

PROVIDER_SEMAPHORES = {
    "openai": asyncio.Semaphore(30),
}

def get_provider(model_name: str) -> str:
    if model_name.startswith("gpt-"):
        return "openai"
    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [10]:
from litellm import acompletion

async def solve_problem(model: str, question: str, num_retries: int = 2) -> GaiaOutput:
    provider = get_provider(model)

    async with PROVIDER_SEMAPHORES[provider]:
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": question},
        ]
        response = await acompletion(
            model=model,
            messages=messages,
            response_format=GaiaOutput,
            num_retries=num_retries,
        )
        
        finish_reason = response.choices[0].finish_reason 
        content = response.choices[0].message.content

        if finish_reason == "refusal" or content is None:
            return GaiaOutput(
                is_solvable=False, 
                unsolvable_reason="Model refused to answer or returned no content.",
                final_answer="",
            )
        
        return GaiaOutput.model_validate_json(content)

In [11]:
def is_correct_answer(prediction: str | None, ground_truth: str | None) -> bool:
    if prediction is None or ground_truth is None:
        return False
    return prediction.strip().lower() == ground_truth.strip().lower()

In [ ]:
async def evaluate_sample(problem: dict, model: str) -> dict:
    question = problem["Question"]
    ground_truth = problem["Final answer"]

    try:

        result = await solve_problem(model, question)
        
        is_correct = is_correct_answer(result.final_answer, ground_truth) if result.is_solvable else False

        return {
            "task_id": problem["task_id"],
            "question": question,
            "ground_truth": ground_truth,
            "is_solvable": result.is_solvable,
            "unsolvable_reason": result.unsolvable_reason,
            "final_answer": result.final_answer,
            "is_correct": is_correct,
        }
    except Exception as e:
        return {
            "task_id": problem["task_id"],
            "question": question,
            "ground_truth": ground_truth,
            "is_solvable": False,
            "unsolvable_reason": str(e),
            "final_answer": "",
            "is_correct": False,
        }

In [22]:
from tqdm.asyncio import tqdm_asyncio

async def run_evaluation(problems: list[dict], model: str) -> list[dict]:
    tasks = [evaluate_sample(problem, model) for problem in problems]
    all_results = await tqdm_asyncio.gather(*tasks)
    return all_results

In [20]:
level_1_problems

Dataset({
    features: ['task_id', 'Question', 'Level', 'Final answer', 'file_name', 'file_path', 'Annotator Metadata'],
    num_rows: 53
})

In [23]:
experiments = await run_evaluation(level_1_problems, model="gpt-5-mini")

  0%|          | 0/53 [00:00<?, ?it/s]Task exception was never retrieved
future: <Task finished name='Task-135' coro=<tqdm_asyncio.gather.<locals>.wrap_awaitable() done, defined at /home/miguel/miniconda3/envs/llm_agent/lib/python3.13/site-packages/tqdm/asyncio.py:80> exception=KeyError('question')>
Traceback (most recent call last):
  File "/home/miguel/miniconda3/envs/llm_agent/lib/python3.13/site-packages/tqdm/asyncio.py", line 82, in wrap_awaitable
    return i, await f
              ^^^^^^^
  File "/tmp/ipykernel_35360/529588857.py", line 2, in evaluate_sample
    question = problem["question"]
               ~~~~~~~^^^^^^^^^^^^
KeyError: 'question'
Task exception was never retrieved
future: <Task finished name='Task-136' coro=<tqdm_asyncio.gather.<locals>.wrap_awaitable() done, defined at /home/miguel/miniconda3/envs/llm_agent/lib/python3.13/site-packages/tqdm/asyncio.py:80> exception=KeyError('question')>
Traceback (most recent call last):
  File "/home/miguel/miniconda3/envs/ll

In [26]:
# count accuracy
num_correct = sum(1 for result in experiments if result["is_correct"])
num_total = len(experiments)
print(f"Accuracy: {num_correct}/{num_total} = {num_correct/num_total:.2%}")

# get solvable and unsolvable counts
num_solvable = sum(1 for result in experiments if result["is_solvable"])
num_unsolvable = num_total - num_solvable
print(f"Solvable: {num_solvable}/{num_total} = {num_solvable/num_total:.2%}")
print(f"Unsolvable: {num_unsolvable}/{num_total} = {num_unsolvable/num_total:.2%}")

Accuracy: 9/53 = 16.98%
Solvable: 15/53 = 28.30%
Unsolvable: 38/53 = 71.70%


In [27]:
# showme some examples of unsolvable problems
unsolvable_examples = [result for result in experiments if not result["is_solvable"]]

for i, example in enumerate(unsolvable_examples[:5]):  # Show the first 5 examples
    print(f"Example {i+1}:")
    print(f"  Question: {example['question']}")
    print(f"  Ground Truth: {example['ground_truth']}")
    print(f"  Unsolvable Reason: {example['unsolvable_reason']}")
    print()

Example 1:
  Question: How many studio albums were published by Mercedes Sosa between 2000 and 2009 (included)? You can use the latest 2022 version of english wikipedia.
  Ground Truth: 3
  Unsolvable Reason: I cannot access the 2022 English Wikipedia from this environment and I do not have full confidence in my memory of Mercedes Sosa's exact studio-release list for 2000–2009, so I cannot provide a guaranteed correct count.

Example 2:
  Question: What was the volume in m^3 of the fish bag that was calculated in the University of Leicester paper "Can Hiccup Supply Enough Fish to Maintain a Dragon’s Diet?"
  Ground Truth: 0.1777
  Unsolvable Reason: I cannot access external websites or retrieve the specific University of Leicester paper; this exact numeric result is not in my stored knowledge up to my cutoff, so I cannot reliably provide the fish bag volume.

Example 3:
  Question: In the video https://www.youtube.com/watch?v=L1vXCYZAYYM, what is the highest number of bird species to b

## TOOLS

In [5]:
from dotenv import load_dotenv, find_dotenv 
load_dotenv(find_dotenv())

True

In [8]:
def calculator(operator: str, a: float, b: float) -> float:
    """
    A simple calculator function that performs basic arithmetic operations.

    Parameters:
    - operator (str): The arithmetic operator ('+', '-', '*', '/').
    - a (float): The first operand.
    - b (float): The second operand.

    Returns:
    - float: The result of the arithmetic operation.

    Raises:
    - ValueError: If an unsupported operator is provided or if division by zero is attempted.
    """
    if operator == '+':
        return a + b
    elif operator == '-':
        return a - b
    elif operator == '*':
        return a * b
    elif operator == '/':
        if b == 0:
            raise ValueError("Division by zero is not allowed.")
        return a / b
    else:
        raise ValueError(f"Unsupported operator: {operator}")

calculator_tools_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "A calculator tool that performs basic arithmetic: addition, subtraction, multiplication and division.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "enum": ["+", "-", "*", "/"],
                    "description": "The arithmetic operator to use.",
                },
                "a": {"type": "number", "description": "The first operand."},
                "b": {"type": "number", "description": "The second operand."},
            },
            "required": ["operator", "a", "b"],
        },
    },
}

In [13]:
from litellm import completion

tools = [calculator_tools_definition]

response_with_tool = completion(    
    model="gpt-5-mini",
    messages=[
        {"role": "user", "content": "What is 1234 x 5678?"},
    ],
    tools=tools,
)

In [14]:
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

None
[ChatCompletionMessageToolCall(function=Function(arguments='{"operator":"*","a":1234,"b":5678}', name='calculator'), id='call_8HpJxrTgh6rjrd0ToUkA47bf', type='function')]


In [17]:
import json

ai_response = response_with_tool.choices[0].message

if ai_response.tool_calls:
    for tool_call in ai_response.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)

        if function_name == "calculator":
            operator = arguments["operator"]
            a = arguments["a"]
            b = arguments["b"]
            result = calculator(operator, a, b)
            print(f"Calculator result: {result}") 

Calculator result: 7006652


## WED SEARCH

In [19]:
import os 
from tavily import TavilyClient

In [20]:
from dotenv import load_dotenv 
load_dotenv()

True

In [21]:
tavily_client = TavilyClient(os.getenv("TAVILY_API_KEY")) 

def search_web(query: str, max_results: int = 2) -> list:  
    response = tavily_client.search(query, max_results=max_results) 
    return response.get("results")

In [22]:
search_web("Kipchoge's marathon world record")

[{'url': 'https://en.wikipedia.org/wiki/Eliud_Kipchoge',
  'title': 'Eliud Kipchoge - Wikipedia',
  'content': 'On 16 September, Kipchoge won the 2018 Berlin Marathon in a time of 2:01:39, breaking the previous world record by 1 minute and 18 seconds (2:02:57 set by fellow countryman Dennis Kimetto at the Berlin Marathon in 2014). It was the greatest improvement in a marathon world record time since 1967. He finished 4:43 min ahead of second-placed fellow Kenyan Amos Kipruto. The world record holder from 2013, Wilson Kipsang of Kenya, came in third at 2:06:48. It was the 11th world record set at the [...] Eliud Kipchoge EGH "Order of the Golden Heart (Kenya)") (born 5 November 1984) is a Kenyan long-distance runner who competes in the marathon and formerly specialized in the 5000 metres. Kipchoge is the 2016 and 2020 Olympic marathon champion, and was the world record holder in the marathon from 2018 to 2023, until that record was broken by Kelvin Kiptum at the 2023 Chicago Marathon. H

In [28]:
def search_web(
    query: str,
    max_results: int = 2,
    topic: str = "general",
    time_range: str | None = None,
    country: str | None = None,
) -> list | str:

    """
    Search the web for a given query.
    """

    try:
        response = tavily_client.search(
            query=query,
            max_results=max_results,
            topic=topic,
            time_range=time_range,
            country=country,
        )
        return response.get("results")
    except Exception as e:
        return f"An error occurred while searching. - {e}"

In [26]:
import inspect


def function_to_input_schema(func) -> dict:
    # Mapea tipos de Python a tipos de JSON Schema (string por defecto)
    type_map = {
        str: "string",
        int: "integer",
        float: "number",
        bool: "boolean",
        list: "array",
        dict: "object",
        type(None): "null",
    }

    # Extrae la firma de la función con el módulo inspect
    try:
        signature = inspect.signature(func)
    except ValueError as e:
        raise ValueError(
            f"Failed to get signature for function {func.__name__}: {str(e)}"
        )

    parameters = {}
    for param in signature.parameters.values():
        try:
            param_type = type_map.get(param.annotation, "string")
        except KeyError as e:
            raise KeyError(
                f"Unknown type annotation {param.annotation} for parameter "
                f"{param.name}: {str(e)}"
            )
        parameters[param.name] = {"type": param_type}

    # Los parámetros sin valor por defecto se marcan como obligatorios
    required = [
        param.name
        for param in signature.parameters.values()
        if param.default == inspect._empty
    ]

    return {
        "type": "object",
        "properties": parameters,
        "required": required,
    }


def format_tool_definition(name: str, description: str, parameters: dict) -> dict:
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": parameters,
        },
    }


def function_to_tool_definition(func) -> dict:
    return format_tool_definition(
        func.__name__, func.__doc__ or "", function_to_input_schema(func)
    )

In [29]:
print(function_to_tool_definition(search_web))

{'type': 'function', 'function': {'name': 'search_web', 'description': '\nSearch the web for a given query.\n', 'parameters': {'type': 'object', 'properties': {'query': {'type': 'string'}, 'max_results': {'type': 'integer'}, 'topic': {'type': 'string'}, 'time_range': {'type': 'string'}, 'country': {'type': 'string'}}, 'required': ['query']}}}
